> 在线讲义：[第 15 章 · 约束与凸优化：拉格朗日乘子、KKT、凸性与 QP](https://wikieden.github.io/math-and-eval-stats/lessons/15-constrained-convex-qp.html) · 本书《数学与实验统计》

# 第 15 章 · 约束与凸优化：拉格朗日乘子、KKT、凸性与 QP

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/15-constrained-convex-qp.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 等式约束 | 3×3 KKT 线性方程组 | `equality_kkt()` | x = y = 0.5，λ = −1 |
| 第二步 不等式约束 | 驻点求 μ | `inequality_kkt()` | x = 1，μ = 2 |
| 第三步 凸性 | P 的特征值 | `is_convex_quadratic(m)` | True / False |
| 第四步 QP | 有效集手算 | `qp_active_set()` | (1.5, 1)，乘子 [1.5, 0] |
| 第四步 核对 | SLSQP | `qp_scipy()` | (1.5, 1) |

例子：x² + y² 在 x + y = 1 上的最小值；(x − 2)² 在 x ≤ 1 上的最小值；一个 2 维 QP（目标点被关节限位挡住）。
里程碑 ME-M11 的带限位 IK 与 MPC 留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
from scipy.optimize import minimize
P_QP = np.diag([1.0, 1.0])
Q_QP = np.array([-3.0, -1.0])  # min ½‖x‖² − 3x₁ − x₂：不受限时最优点 (3, 1)
G_QP = np.array([[1.0, 0.0], [0.0, 1.0]])
H_QP = np.array([1.5, 2.0])  # 限位 x₁ ≤ 1.5、x₂ ≤ 2

**函数 `equality_kkt`**：min x² + y²  s.t. x + y = 1：拉格朗日 L = x² + y² + λ(x + y − 1)，∇L = 0 与约束连立成线性方程组。返回 (x, y, λ)。

In [ ]:
def equality_kkt():
    """min x² + y²  s.t. x + y = 1：拉格朗日 L = x² + y² + λ(x + y − 1)，∇L = 0 与约束连立成线性方程组。返回 (x, y, λ)。"""
    k = np.array([[2.0, 0.0, 1.0], [0.0, 2.0, 1.0], [1.0, 1.0, 0.0]])
    rhs = np.array([0.0, 0.0, 1.0])
    return tuple(np.linalg.solve(k, rhs))

**函数 `inequality_kkt`**：min (x − 2)²  s.t. x ≤ 1：约束起作用，x = 1；驻点条件 2(x − 2) + μ = 0 → μ = 2 ≥ 0。返回 (x, μ)。

In [ ]:
def inequality_kkt() -> tuple[float, float]:
    """min (x − 2)²  s.t. x ≤ 1：约束起作用，x = 1；驻点条件 2(x − 2) + μ = 0 → μ = 2 ≥ 0。返回 (x, μ)。"""
    x = 1.0
    return x, -2 * (x - 2)

**函数 `is_convex_quadratic`**：二次函数 ½xᵀMx 凸 ⇔ M 半正定（特征值都 ≥ 0）。

In [ ]:
def is_convex_quadratic(m: np.ndarray) -> bool:
    """二次函数 ½xᵀMx 凸 ⇔ M 半正定（特征值都 ≥ 0）。"""
    return bool(np.all(np.linalg.eigvalsh(m) >= -1e-12))

**函数 `qp_active_set`**：小 QP 的手算解：先不管约束得 (3, 1)，x₁ 超了限位 → 把 x₁ ≤ 1.5 当等式解 KKT。返回 (x, 乘子)。

In [ ]:
def qp_active_set() -> tuple[np.ndarray, np.ndarray]:
    """小 QP 的手算解：先不管约束得 (3, 1)，x₁ 超了限位 → 把 x₁ ≤ 1.5 当等式解 KKT。返回 (x, 乘子)。"""
    k = np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0]])
    rhs = np.array([3.0, 1.0, 1.5])
    sol = np.linalg.solve(k, rhs)
    return sol[:2], np.array([sol[2], 0.0])

**函数 `qp_scipy`**：同一个 QP 交给 scipy 的 SLSQP 求解，用来核对。

In [ ]:
def qp_scipy() -> np.ndarray:
    """同一个 QP 交给 scipy 的 SLSQP 求解，用来核对。"""
    f = lambda x: 0.5 * x @ P_QP @ x + Q_QP @ x
    cons = {"type": "ineq", "fun": lambda x: H_QP - G_QP @ x}
    return minimize(f, np.zeros(2), constraints=[cons], method="SLSQP").x

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 等式约束：拉格朗日乘子")
x, y, lam = equality_kkt()
print(f"  min x² + y²，x + y = 1：x = {x:.2f}，y = {y:.2f}，λ = {lam:.2f}（f 的梯度 [{2 * x:.1f}, {2 * y:.1f}] 和约束法向 [1, 1] 平行）")

In [ ]:
print("\n第二步 不等式约束：KKT 与互补松弛")
x, mu = inequality_kkt()
print(f"  min (x − 2)²，x ≤ 1：x = {x}，μ = {mu}（μ ≥ 0、约束取等号、μ·(x − 1) = 0）")

In [ ]:
print("\n第三步 凸性：Hessian 半正定")
for m in (np.diag([1.0, 10.0]), np.array([[1.0, 2.0], [2.0, 1.0]])):
    print(f"  {m.tolist()}：特征值 {np.linalg.eigvalsh(m).tolist()}，凸 = {is_convex_quadratic(m)}")

In [ ]:
print("\n第四步 QP 标准形：min ½xᵀPx + qᵀx  s.t. Gx ≤ h")
xs, lam = qp_active_set()
print(f"  不受限的最优点 (3, 1)；加限位 x₁ ≤ 1.5、x₂ ≤ 2 → 手算 x = {xs.tolist()}，乘子 {lam.tolist()}")
print(f"  scipy SLSQP：x = {np.round(qp_scipy(), 4).tolist()}")

## 3. 代码里才看得到的

1. **KKT 矩阵的结构**：左上是目标的 Hessian，右上 / 左下是约束的雅可比，右下是 0。它不是正定的（是"鞍点矩阵"），不能用 Cholesky 解。
2. **scipy 的不等式约定是 ≥ 0**：要表达 Gx ≤ h，就写 `h − Gx ≥ 0`；OSQP 又是 l ≤ Ax ≤ u。换求解器时先查约定。
3. **λ 的符号取决于写法**：L = f + λh 时 λ = −1；写成 L = f − λh 时 λ = +1。只要前后一致，敏感度的意义不变。

## 4. 自己改一改

1. 把 QP 的限位改成 x₁ ≤ 1.5、x₂ ≤ 0.5，两个约束都起作用，手算乘子，再用 `qp_scipy` 核对。
2. 把等式约束改成 x + y = 2，λ 变成多少？最优值变了多少，和 −λ 对得上吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 山路最低点，坡度垂直于路，梯度和法向平行；
> λ 是影子价格，约束放松一点赚多少；
> 墙边球：可行、μ 非负、驻点、互补松弛；
> 凸则局部即全局，限位 IK、MPC、全身控制都是 QP。